# Round 2: 1000G-referenced Mahalanobis filter

Round 1 (`01_premade_label_filter.ipynb`) picks a base cohort from AoU's premade label, no PCA. This notebook fits a Mahalanobis ellipsoid against a genuine 1000G reference population's mean/covariance (CEU+GBR for EUR, pooled AFR superpop for `afr`), not a self-referential fit. Different `THRESHOLD_QUANTILE`s per `SAMPLE_SET`; `eur_premade_label` skips the ellipsoid.

Reads `04_build_ancestry_panel_hm3.ipynb`'s HM3-restricted panel, not the GRM panel -- intersecting the GRM panel (randomly thinned, unrelated to HM3) against 1000G left only ~10K variants; the HM3 panel should land much closer to `1kg_all_pruned.prune.in`'s count.

Writes `final_keep_ids_{SAMPLE_SET}_{prob_tag}.txt` to `ancestry_panel_{BASE_GROUP}/final_pca/`.

## Compute resource

8-16 vCPU is plenty -- lighter than GRM construction.

## Setup

plink2: manual install, same pattern as everywhere else in this repo.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version
nproc
free -h

In [ ]:
import os

bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Sample set configuration

Each `SAMPLE_SET` maps to a `BASE_GROUP`, an `ellipsoid_ref_pops`, and a threshold. `None` = skip the ellipsoid, keep the whole premade-label cohort.

In [ ]:
SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
}

SAMPLE_SETS = {
    "eur":               {"base_group": "eur", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": 0.999999},
    "eur_stringent":     {"base_group": "eur", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": 0.99},
    "eur_loose":         {"base_group": "eur", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": 0.99999999},
    "eur_premade_label": {"base_group": "eur", "ellipsoid_ref_pops": ["CEU", "GBR"], "ellipsoid_threshold": None},
    "afr":               {"base_group": "afr", "ellipsoid_ref_pops": SUPERPOP_POPS["AFR"], "ellipsoid_threshold": 0.999},
}

N_PCS_ELLIPSOID = 2   # Mahalanobis fit dimensionality, matching every prior round's own convention

CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "covariance_v9"
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
KG_DIR = f"{WORKSPACE_BUCKET}/1000g_reference"   # 02_build_1000g_reference.ipynb's output, CDR-independent
PANEL_PATH = f"{KG_DIR}/integrated_call_samples_v3.20130502.ALL.panel"
KG_OUT_PREFIX = f"{KG_DIR}/1kg_all_qc"
KG_PRUNE_PREFIX = f"{KG_DIR}/1kg_all_pruned"
KG_PCA_PREFIX = f"{KG_DIR}/1kg_all_pca"
for ext in ("eigenvec.allele", "acount"):
    assert os.path.isfile(f"{KG_PCA_PREFIX}.{ext}"), (
        f"missing {KG_PCA_PREFIX}.{ext} -- run 02_build_1000g_reference.ipynb first"
    )

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_round2")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)

## Inputs (per `BASE_GROUP`)

`04_build_ancestry_panel_hm3.ipynb`'s merged HM3 panel -- already `--keep`-restricted to round 1's cohort.

In [ ]:
BASE_GROUP = "eur"   # <-- change this and rerun for "eur" and "afr"

PANEL_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_panel_{BASE_GROUP}"
MERGED_NAME = f"ancestry_panel_hm3_{CDR_VERSION}_{BASE_GROUP}"
MERGED_PREFIX = os.path.join(LOCAL_WORK_DIR, MERGED_NAME)

for ext in ("pgen", "pvar", "psam"):
    bucket_path = f"{PANEL_DIR}/{MERGED_NAME}.{ext}"
    local_path = f"{MERGED_PREFIX}.{ext}"
    assert os.path.isfile(bucket_path), (
        f"missing merged ancestry panel: {bucket_path!r} -- run 04_build_ancestry_panel_hm3.ipynb's "
        f"merge section for BASE_GROUP={BASE_GROUP!r} first"
    )
    if not os.path.isfile(local_path) or os.path.getsize(local_path) != os.path.getsize(bucket_path):
        import shutil
        shutil.copy(bucket_path, local_path)

FINAL_PCA_BUCKET_DIR = f"{PANEL_DIR}/final_pca"
os.makedirs(FINAL_PCA_BUCKET_DIR, exist_ok=True)

PROJECT_PREFIX = os.path.join(LOCAL_WORK_DIR, f"acaf_agreeing_{BASE_GROUP}")
PROJECTED_OUT = os.path.join(LOCAL_WORK_DIR, f"acaf_projected_{BASE_GROUP}")

print(MERGED_PREFIX)
print(FINAL_PCA_BUCKET_DIR)

## Harmonize variants against the 1000G reference

Biallelic-filter, then build an "agreeing" SNP list (ID + REF/ALT match against `.acount`) -- a bare ID match would silently let `--score` skip allele mismatches.

In [ ]:
%%bash -s "$MERGED_PREFIX" "$KG_PCA_PREFIX" "$PROJECT_PREFIX"
set -e
MERGED_PREFIX=$1
KG_PCA_PREFIX=$2
PROJECT_PREFIX=$3

BIALLELIC_PREFIX="${PROJECT_PREFIX}_biallelic"

plink2   --pfile "$MERGED_PREFIX"   --max-alleles 2   --rm-dup exclude-all   --make-pgen   --out "$BIALLELIC_PREFIX"

grep -v '^##' "${BIALLELIC_PREFIX}.pvar" | awk 'NR>1 {print $3, $4, $5}' | sort > "${PROJECT_PREFIX}_acaf_id_ref_alt.sorted"
awk 'NR>1 {print $2, $3, $4}' "${KG_PCA_PREFIX}.acount" | sort > "${PROJECT_PREFIX}_acount_id_ref_alt.sorted"
comm -12 "${PROJECT_PREFIX}_acaf_id_ref_alt.sorted" "${PROJECT_PREFIX}_acount_id_ref_alt.sorted" | awk '{print $1}' > "${PROJECT_PREFIX}_agreeing_snps.ids"

echo "Biallelic panel variants: $(grep -v '^##' "${BIALLELIC_PREFIX}.pvar" | tail -n +2 | wc -l)"
echo "Agreeing with 1000G reference (ID+REF+ALT): $(wc -l < "${PROJECT_PREFIX}_agreeing_snps.ids")"

## Project the round-1 cohort onto the 1000G PCs

`--score` against the 1000G allele weights, AVG scoring + `list-variants`.

In [ ]:
%%bash -s "$PROJECT_PREFIX" "$KG_PCA_PREFIX" "$PROJECTED_OUT"
set -e
PROJECT_PREFIX=$1
KG_PCA_PREFIX=$2
PROJECTED_OUT=$3

BIALLELIC_PREFIX="${PROJECT_PREFIX}_biallelic"
AGREEING_SNPS="${PROJECT_PREFIX}_agreeing_snps.ids"

WEIGHTS="${KG_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'PC20' | cut -d: -f1)

plink2   --pfile "$BIALLELIC_PREFIX"   --extract "$AGREEING_SNPS"   --nonfounders   --read-freq "${KG_PCA_PREFIX}.acount"   --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize list-variants   --score-col-nums "${PC1_COL}-${PC_LAST_COL}"   --out "$PROJECTED_OUT"

head "${PROJECTED_OUT}.sscore"
wc -l "${PROJECTED_OUT}.sscore.vars"

## Reproject the 1000G reference onto its own PCs

Sanity check, restricted to the same variant set the AoU projection used. Expect high |correlation| per PC (sign is arbitrary).

In [ ]:
%%bash -s "$KG_OUT_PREFIX" "$KG_PCA_PREFIX" "$PROJECTED_OUT"
set -e
KG_OUT_PREFIX=$1
KG_PCA_PREFIX=$2
PROJECTED_OUT=$3

REPROJECT_OUT="${PROJECTED_OUT}_kg_reprojected"
USED_VARS="${PROJECTED_OUT}.sscore.vars"

WEIGHTS="${KG_PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '	' '
' | grep -nx 'PC20' | cut -d: -f1)

plink2   --bfile "$KG_OUT_PREFIX"   --extract "$USED_VARS"   --nonfounders   --read-freq "${KG_PCA_PREFIX}.acount"   --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize   --score-col-nums "${PC1_COL}-${PC_LAST_COL}"   --out "$REPROJECT_OUT"

head "${REPROJECT_OUT}.sscore"

In [ ]:
import pandas as pd
import numpy as np

direct = pd.read_csv(f"{KG_PCA_PREFIX}.eigenvec", sep=r"\s+")
reproj = pd.read_csv(f"{PROJECTED_OUT}_kg_reprojected.sscore", sep=r"\s+")

direct_id_col = "#IID" if "#IID" in direct.columns else "IID"
merged = direct.merge(reproj, left_on=direct_id_col, right_on="IID", suffixes=("_direct", "_reproj"))

score_cols = [c for c in reproj.columns if c.startswith("PC") and c.endswith("_AVG")]
for i, pc in enumerate(range(1, 21)):
    direct_col = f"PC{pc}"
    score_col = score_cols[i] if i < len(score_cols) else None
    if direct_col in merged.columns and score_col in merged.columns:
        corr = merged[direct_col].corr(merged[score_col])
        print(f"PC{pc}: r = {corr:.4f}")

## Plot the projections

AoU alone, and AoU overlaid on the reprojected 1000G reference (by `pop`) -- visual check before trusting any threshold.

In [ ]:
import matplotlib.pyplot as plt

# Loaded independently here (not reused from the ellipsoid cell below) so this
# plot can run right after projection/reprojection, without depending on the
# ellipsoid cell having executed first.
aou_proj_plot = pd.read_csv(f"{PROJECTED_OUT}.sscore", sep=r"\s+")
aou_id_col_plot = "#IID" if "#IID" in aou_proj_plot.columns else "IID"

ref_proj_plot = pd.read_csv(f"{PROJECTED_OUT}_kg_reprojected.sscore", sep=r"\s+")
ref_id_col_plot = "#IID" if "#IID" in ref_proj_plot.columns else "IID"
panel_plot = pd.read_csv(PANEL_PATH, sep="\t")
ref_proj_plot = ref_proj_plot.rename(columns={ref_id_col_plot: "sample"}).merge(
    panel_plot[["sample", "pop", "super_pop"]], on="sample"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: AoU alone
axes[0].scatter(aou_proj_plot["PC1_AVG"], aou_proj_plot["PC2_AVG"], s=4, alpha=0.2, color="tab:blue")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_title(f"AoU {BASE_GROUP} cohort, projected onto 1000G PC space (n={len(aou_proj_plot):,})")

# Panel 2: AoU + reprojected 1000G reference, colored by pop
for pop, grp in ref_proj_plot.groupby("pop"):
    axes[1].scatter(grp["PC1_AVG"], grp["PC2_AVG"], s=8, alpha=0.7, label=pop)
axes[1].scatter(aou_proj_plot["PC1_AVG"], aou_proj_plot["PC2_AVG"], s=3, alpha=0.1, color="black", label=f"AoU {BASE_GROUP}")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("AoU cohort vs. reprojected 1000G reference")
axes[1].legend(markerscale=3, fontsize=8, loc="best")

plt.tight_layout()
plt.show()

## Mahalanobis ellipsoid filter (round 2)

Fit on `ellipsoid_ref_pops`' mean/covariance, score every AoU sample, write one keep-list per `SAMPLE_SET`. `eur_premade_label` keeps everyone.

In [ ]:
from scipy.stats import chi2

panel = pd.read_csv(PANEL_PATH, sep="\t")

aou_proj = pd.read_csv(f"{PROJECTED_OUT}.sscore", sep=r"\s+")
aou_id_col = "#IID" if "#IID" in aou_proj.columns else "IID"
aou_proj = aou_proj.rename(columns={aou_id_col: "sample"})

ref_proj = pd.read_csv(f"{PROJECTED_OUT}_kg_reprojected.sscore", sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref_proj.columns else "IID"
ref_proj = ref_proj.rename(columns={ref_id_col: "sample"}).merge(panel[["sample", "pop", "super_pop"]], on="sample")

pc_cols_mahal = [f"PC{i}_AVG" for i in range(1, N_PCS_ELLIPSOID + 1)]


def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)


for sample_set, cfg in SAMPLE_SETS.items():
    if cfg["base_group"] != BASE_GROUP:
        continue
    threshold_quantile = cfg["ellipsoid_threshold"]

    if threshold_quantile is None:
        keep_ids = aou_proj[aou_id_col if aou_id_col in aou_proj.columns else "sample"]
        prob_tag = "unfiltered"
        keep_ids = aou_proj["sample"]
    else:
        ref_pop_list = cfg["ellipsoid_ref_pops"]
        ref_mask = ref_proj["pop"].isin(ref_pop_list)
        ref_pcs = ref_proj.loc[ref_mask, pc_cols_mahal].values
        assert len(ref_pcs) > N_PCS_ELLIPSOID, f"too few reference samples ({len(ref_pcs)}) for {sample_set}"

        mean = ref_pcs.mean(axis=0)
        cov_inv = np.linalg.inv(np.cov(ref_pcs, rowvar=False))
        threshold = np.sqrt(chi2.ppf(threshold_quantile, df=N_PCS_ELLIPSOID))

        aou_mahal = np.array([mahal(row, mean, cov_inv) for row in aou_proj[pc_cols_mahal].values])
        keep_ids = aou_proj.loc[aou_mahal <= threshold, "sample"]
        prob_tag = f"p{threshold_quantile * 100:g}"

        ref_retention = ref_proj.loc[ref_mask, pc_cols_mahal].apply(
            lambda row: mahal(row.values, mean, cov_inv) <= threshold, axis=1
        ).mean()
        print(f"[{sample_set}] {'+'.join(ref_pop_list)} reference retention: {ref_retention:.1%}")

    keep_path = os.path.join(FINAL_PCA_BUCKET_DIR, f"final_keep_ids_{sample_set}_{prob_tag}.txt")
    keep_ids.to_csv(keep_path, index=False, header=False)
    print(f"[{sample_set}] {len(keep_ids)}/{len(aou_proj)} retained -> {keep_path}")

## Next steps

`06_final_pca.ipynb` reads these keep-lists and refits to 10 PCs within each `SAMPLE_SET`.